In [9]:
import serial, time

ser = serial.Serial('COM5', baudrate=115200, timeout=1)
time.sleep(0.5)

def send(cmd):
    ser.write((cmd + "\n").encode())
    time.sleep(0.05)
    r = ser.readline().decode(errors="ignore").strip()
    return r

# Identify
print(send("*IDN?"))

# ----- Channel 1 -----
send("CHANNEL 1")
send("FREQ:CW 123.456789MHZ")
send("POWER -12")
# send("OUTP:STAT ON")
print("CH1 Freq:", send("FREQ:QUANT?"))
print("CH1 Power:", send("POWER?"))

# ----- Channel 2 -----
send("CHANNEL 2")
send("FREQ:CW 56.789012MHZ")
send("POWER -12")
# send("OUTP:STAT ON")
print("CH2 Freq:", send("FREQ:QUANT?"))
print("CH2 Power:", send("POWER?"))
print(send("FREQ:MAX?"))
ser.close()


DS Instruments,SG6000X,4134,V15.47
CH1 Freq: 123456807HZ
CH1 Power: -12.00dBm
CH2 Freq: 56788995HZ
CH2 Power: -12.00dBm
6000000000


In [1]:
# for SG12000PRO

import socket

def find_dsi_devices(timeout=2):
    port = 30718
    packet = bytes([0x00, 0x00, 0x00, 0xF6])

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
    sock.settimeout(timeout)

    sock.bind(("", 0))
    sock.sendto(packet, ("255.255.255.255", port))

    devices = []

    try:
        while True:
            data, addr = sock.recvfrom(1024)
            print("Found device at:", addr[0])
            devices.append(addr[0])
    except socket.timeout:
        pass
    finally:
        sock.close()

    return devices

devices = find_dsi_devices()

Found device at: 192.168.168.40


In [2]:
import socket

SG_IP = "192.168.168.40"   # replace with your SG12000PRO IP
SG_PORT = 10001

sock = None


def connect():
    global sock

    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(2.0)
    sock.connect((SG_IP, SG_PORT))

    print(f"Connected to {SG_IP}:{SG_PORT}")


def disconnect():
    global sock

    if sock is not None:
        sock.close()
        sock = None

    print("Disconnected")


def write(command):
    if sock is None:
        raise RuntimeError("Not connected")

    message = command.strip() + "\r\n"
    sock.sendall(message.encode("ascii"))


def query(command):
    write(command)

    response = sock.recv(1024)
    return response.decode("ascii").strip()


def identify():
    return query("*IDN?")


def set_frequency(freq_mhz):
    write(f"FREQ:CW {freq_mhz}MHZ")


def get_frequency():
    return query("FREQ:CW?")


def set_power(power_dbm):
    write(f"POWER {power_dbm}")


def get_power():
    return query("POWER?")


def output_on():
    write("OUTP:STAT ON")


def output_off():
    write("OUTP:STAT OFF")


def get_output():
    return query("OUTP:STAT?")

In [7]:
connect()

print(identify())

Connected to 192.168.168.40:10001
DS Instruments,SG12000PRO,5059,V8.54


In [5]:
print("Frequency:", get_frequency())
print("Power:", get_power())
print("Output:", get_output())

Frequency: 630000000HZ
Power: +0.00dBm
Output: OFF


In [8]:
set_frequency(8037.75032)

In [9]:
disconnect()

Disconnected
